# Data Restructuring & Saving

In this notebook, we will:
- Modify the **Spike Path DataFrame** using metadata dates.
- Add unique **hashes** for efficient storage and quick lookup of:
  - Sequence paths
  - Mutation paths

This restructuring will ensure:
- Consistent alignment of spike data with external metadata timelines.
- Fast and reliable retrieval of sequence/mutation information.
- A standardized format for downstream analysis and storage.

In [ ]:
# Data handling
import pandas as pd
from datetime import datetime
import pickle
from tqdm import tqdm

# Sequence parsing
from Bio import SeqIO

# --- Custom imports ---
from Helpers import *   # Contains utility functions for data processing and plotting

In [3]:
# --- Load reference sequence and spike paths data ---

# Load reference spike sequence
refSeq = getReference("/data/sukhanna/MPP/cov_reference.fa")

# Load spike paths DataFrame
spike_paths_df = pd.read_pickle("/data/sukhanna/MPP/ver_2/SpikePaths_231215.pkl") #Takes some time to load ~ 5 mins | Switch to parquet 

# Quick sanity checks
print(f"Reference sequence length: {len(refSeq)} amino acids")
print(f"Spike paths data: {spike_paths_df.shape[0]} rows × {spike_paths_df.shape[1]} columns")

# Preview spike paths DataFrame
spike_paths_df.head()

Reference sequence length: 1273 amino acids
Spike paths data: 15906218 rows × 4 columns


,sample_id,parsed_spike_list,node_count,mutation_count
0,England/LIVE-9EBF7/2020|OX592862.1|2020-04-04,[[S:A831V]],1,1
1,England/SHEF-D10CC/2020|OY361628.1|2020-03-24,[[S:D1118Y]],1,1
2,England/20156050304/2020|EPI_ISL_466336|2020-0...,[[S:P295S]],1,1
3,England/PHEC-1E01E/2020|OY305731.1|2020-04-03,[[S:P295S]],1,1
4,England/LIVE-A9C05/2020|OX648715.1|2020-03-24,[[S:P681S]],1,1


In [4]:
# --- Extract dates from sample_id and report parsing failures ---

extracted_dates = []
failed_rows = []

for i, name in enumerate(spike_paths_df['sample_id']):
    try:
        parts = name.split('|')
        date_str = parts[-1]
        parsed_date = datetime.strptime(date_str, "%Y-%m-%d")
        extracted_dates.append(parsed_date)
    except Exception:
        extracted_dates.append(pd.NaT)
        failed_rows.append((i, name))

spike_paths_df['Date'] = extracted_dates

# Report parsing results
total = len(spike_paths_df)
fail_count = len(failed_rows)
fail_pct = 100 * fail_count / total

print(f"Total entries: {total}")
print(f"Failed to parse dates: {fail_count} ({fail_pct:.2f}%)")

Total entries: 15906218
Failed to parse dates: 254277 (1.60%)


In [5]:
# --- Sort spike_paths_df by Date ascending ---

spike_paths_df = spike_paths_df.sort_values(by='Date', ascending=True).reset_index(drop=True)

# --- Define date range to filter ---

start_date = pd.Timestamp('2020-01-01')
end_date = pd.Timestamp('2023-12-31')

# Filter rows to keep only those within the date range
spike_paths_df = spike_paths_df[
    (spike_paths_df['Date'] >= start_date) & (spike_paths_df['Date'] <= end_date)
].reset_index(drop=True)

# Morphing Data: Creating Hashed Mutation and Sequence IDs

In this step, we will generate unique hash identifiers for:

- **Mutation sets**: to create compact Mutation IDs representing each unique mutation combination.
- **Sequences**: to create Sequence IDs that enable efficient storage and quick lookups.

This hashing significantly reduces data size and improves performance of downstream operations such as:

- Data merging and grouping.
- Fast mutation and sequence comparisons.
- Storage and retrieval optimizations.

The hash IDs serve as stable keys that abstract away verbose mutation or sequence details.

In [8]:
# --- Flatten nested mutation lists in parsed_spike_list ---

def flatten_nested_list(nested_list):
    """
    Flatten a nested list of mutations into a single flat list.

    Args:
        nested_list (list of lists): Nested list to flatten.

    Returns:
        list: Flattened list containing all items.
    """
    if not nested_list:
        return []
    return [item for sublist in nested_list for item in sublist]

# Apply the flattening function to the 'parsed_spike_list' column
spike_paths_df['parsed_spike_list'] = spike_paths_df['parsed_spike_list'].apply(flatten_nested_list)

In [9]:
from tqdm import tqdm
tqdm.pandas(desc="Applying path mutations")

def apply_path_mutations(path, refSeq):
    """
    Apply a list of mutations (path) to the reference sequence.

    Args:
        path (list): List of mutation strings.
        refSeq (str or Seq): Reference sequence.
    
    Returns:
        str: Mutated sequence string.
    """
    if not path:  # empty or None path
        return str(refSeq)
    return apply_mutations_flat_list(path, refSeq)

# Cache dictionary to avoid repeated computations
cache = {}

def mutate_row_cached(path):
    """
    Mutate a sequence for a given mutation path using caching for speed.

    Args:
        path (list): List of mutation strings representing mutation path.

    Returns:
        str: Mutated sequence string.
    """
    key = tuple(path)  # immutable key for caching
    if key not in cache:
        cache[key] = apply_path_mutations(path, refSeq)
    return cache[key]

# Apply mutation paths with progress bar, store results in new column
spike_paths_df['mutated_sequence'] = spike_paths_df['parsed_spike_list'].progress_apply(mutate_row_cached)


Applying path mutations: 100%|██████████| 15651935/15651935 [01:04<00:00, 242055.68it/s]


# Generating Consistent Mutation and Sequence Keys

To facilitate fast and reliable lookups, we create unique keys for each:

- **Mutation set**:  
  Each unique combination of mutations is converted into a consistent string, hashed, and prefixed with `M_` to form a compact Mutation Key.

- **Sequence**:  
  Each mutated amino acid sequence is hashed and prefixed with `S_` to generate a unique Sequence Key.

These keys provide:  
- Fixed-length identifiers for easy indexing and storage.  
- Consistency across datasets to avoid redundancy.  
- Improved efficiency for join and comparison operations.

In [10]:
# --- Generate Mutation_ID and Sequence_ID columns ---

# Convert parsed spike mutation lists to tuples for hashing and uniqueness
spike_paths_df['mutation_path_tuple'] = spike_paths_df['parsed_spike_list'].apply(tuple)

# Create a unique list of mutation paths and assign Mutation_IDs
unique_paths = spike_paths_df['mutation_path_tuple'].unique()
mutation_id_map = {path: f"M_{i}" for i, path in enumerate(unique_paths)}
spike_paths_df['Mutation_ID'] = spike_paths_df['mutation_path_tuple'].apply(lambda x: mutation_id_map[x])

# Create a unique list of mutated sequences and assign Sequence_IDs
unique_seqs = spike_paths_df['mutated_sequence'].unique()
sequence_id_map = {seq: f"S_{i}" for i, seq in enumerate(unique_seqs)}
spike_paths_df['Sequence_ID'] = spike_paths_df['mutated_sequence'].apply(lambda x: sequence_id_map[x])

In [12]:
# --- Create final_df and reverse lookup maps ---

final_df = spike_paths_df[['sample_id', 'Mutation_ID', 'Sequence_ID', 'Date']].copy()
final_df.rename(columns={'sample_id': 'Sample_ID'}, inplace=True)

# Reverse lookup dictionaries for sequences and mutation paths
sequence_dict = {v: k for k, v in sequence_id_map.items()}
mutation_path_dict = {v: k for k, v in mutation_id_map.items()}

In [13]:
# --- Preview of final_df ---
print(f"final_df shape: {final_df.shape}")
final_df.head()

final_df shape: (15651935, 4)


,Sample_ID,Mutation_ID,Sequence_ID,Date
0,TrinidadandTobago/3074/2020|EPI_ISL_16253520|2...,M_0,S_0,2020-01-01
1,TrinidadandTobago/3851/2020|EPI_ISL_16253526|2...,M_0,S_0,2020-01-01
2,TrinidadandTobago/3753/2020|EPI_ISL_16253524|2...,M_0,S_0,2020-01-01
3,Canada/BC-BCCDC-13444/2020|EPI_ISL_2522129|202...,M_1,S_1,2020-01-01
4,Canada/BC-BCCDC-13443/2020|EPI_ISL_2522128|202...,M_1,S_1,2020-01-01


In [ ]:
# Execute Cell to Save the Structures
# import pickle
# from pathlib import Path

# # --- Define output directory and paths ---
# output_dir = Path("/data/sukhanna/MPP/ver_2/ver_2A/")
# output_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

# final_df_path = output_dir / "final_df_meta.parquet"
# sequence_dict_path = output_dir / "sequence_dict_meta.pkl"
# mutation_path_dict_path = output_dir / "mutation_path_dict_meta.pkl"

# # --- Save DataFrame as Parquet ---
# final_df.to_parquet(final_df_path, compression="snappy", index=False)

# # --- Save dictionaries as pickle files (no compression) ---
# with open(sequence_dict_path, "wb") as f_seq:
#     pickle.dump(sequence_dict, f_seq, protocol=pickle.HIGHEST_PROTOCOL)

# with open(mutation_path_dict_path, "wb") as f_mut:
#     pickle.dump(mutation_path_dict, f_mut, protocol=pickle.HIGHEST_PROTOCOL)

# print(f"Saved final_df to {final_df_path}")
# print(f"Saved sequence_dict to {sequence_dict_path}")
# print(f"Saved mutation_path_dict to {mutation_path_dict_path}")